# Selective KV Cache Loading Test

This notebook tests the selective KV cache loading modifications to LMCache and vLLM.

**What it does:**
1. Content-only hashing (no prefix chain)
2. UUID-based block retrieval
3. Skip missing blocks instead of invalidating all
4. Contiguous slot mapping for selective loads

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Clone Modified Repos

Upload the modified files or clone from your fork.

In [ ]:
# Option A: Clone from GitHub (if you pushed the changes)
# !git clone https://github.com/YOUR_USERNAME/LMCache.git
# !git clone https://github.com/YOUR_USERNAME/vllm.git

# Option B: Clone original and apply patches
!git clone https://github.com/LMCache/LMCache.git
!git clone https://github.com/vllm-project/vllm.git

## 3. Apply Modifications

Apply the selective loading changes to LMCache and vLLM.

In [ ]:
%%writefile LMCache/lmcache/v1/token_database_patch.py
"""
Patch for token_database.py - Content-only hashing

Find the _prefix_hash method and replace it with this version.
"""

# In _prefix_hash method, change from:
#
# def _prefix_hash(self, token_chunks):
#     prefix_hash = self._get_init_hash()
#     for token_chunk in token_chunks:
#         prefix_hash = self._hash_tokens(token_chunk, prefix_hash)  # CHAINED!
#         yield prefix_hash
#
# To:
#
# def _prefix_hash(self, token_chunks):
#     # MODIFIED: Content-only hashing (no prefix chain)
#     for token_chunk in token_chunks:
#         chunk_hash = self._hash_tokens(token_chunk)  # No prefix!
#         yield chunk_hash

print("See comments above for the patch to apply")

In [ ]:
# Apply the token_database.py patch
import re

with open('LMCache/lmcache/v1/token_database.py', 'r') as f:
    content = f.read()

# Find and replace _prefix_hash method
old_pattern = r'''def _prefix_hash\(
        self,
        token_chunks: Iterable\[Union\[torch\.Tensor, List\[int\]\]\],
    \) -> Iterable\[int\]:
        """.*?""".*?
        prefix_hash = self\._get_init_hash\(\)
        for token_chunk in token_chunks:
            prefix_hash = self\._hash_tokens\(token_chunk, prefix_hash\)
            yield prefix_hash'''

new_code = '''def _prefix_hash(
        self,
        token_chunks: Iterable[Union[torch.Tensor, List[int]]],
    ) -> Iterable[int]:
        """
        MODIFIED: Content-only hashing (no prefix chain).
        This allows selective block loading without requiring all previous blocks.
        """
        for token_chunk in token_chunks:
            chunk_hash = self._hash_tokens(token_chunk)  # No prefix dependency!
            yield chunk_hash'''

# Simple replacement for the key line
content = content.replace(
    'prefix_hash = self._hash_tokens(token_chunk, prefix_hash)',
    'prefix_hash = self._hash_tokens(token_chunk)  # MODIFIED: content-only'
)

with open('LMCache/lmcache/v1/token_database.py', 'w') as f:
    f.write(content)

print("Patched token_database.py for content-only hashing")

In [ ]:
# Add hashes/offsets parameters to process_tokens in token_database.py
with open('LMCache/lmcache/v1/token_database.py', 'r') as f:
    content = f.read()

# Add import for List if not present
if 'from typing import' in content and 'List' not in content.split('from typing import')[1].split('\n')[0]:
    content = content.replace('from typing import', 'from typing import List, ')

# Find process_tokens and add hashes/offsets parameters
old_sig = 'def process_tokens(\n        self,\n        tokens:'
new_sig = '''def process_tokens(
        self,
        tokens: Optional[Union[torch.Tensor, List[int]]] = None,
        hashes: Optional[List[int]] = None,
        offsets: Optional[List[int]] = None,'''

if 'hashes: Optional[List[int]]' not in content:
    # Add after the method signature
    content = content.replace(
        'def process_tokens(\n        self,\n        tokens: Union[torch.Tensor, List[int]],',
        new_sig
    )

with open('LMCache/lmcache/v1/token_database.py', 'w') as f:
    f.write(content)

print("Added hashes/offsets parameters")

## 4. Install Dependencies

In [ ]:
# Install LMCache
%cd LMCache
!pip install -e . -q
%cd ..

In [ ]:
# Install vLLM
!pip install vllm -q

## 5. Test Content-Only Hashing

In [ ]:
import torch
from lmcache.v1.token_database import ChunkedTokenDatabase
from lmcache.v1.config import LMCacheEngineConfig

def test_content_only_hashing():
    """
    Test that same content produces same hash, regardless of prefix.
    """
    print("="*60)
    print("TEST: Content-Only Hashing")
    print("="*60)

    cfg = LMCacheEngineConfig.from_legacy(chunk_size=256, backend="cpu")
    db = ChunkedTokenDatabase(cfg, None)

    # Same target chunk
    target_chunk = torch.tensor([100] * 256)

    # Different prefixes
    prefix_a = torch.tensor([1] * 256)
    prefix_b = torch.tensor([2] * 256)

    tokens_a = torch.cat([prefix_a, target_chunk])
    tokens_b = torch.cat([prefix_b, target_chunk])

    # Process both
    results_a = list(db.process_tokens(tokens=tokens_a))
    results_b = list(db.process_tokens(tokens=tokens_b))

    # Get second chunk hash from each
    hash_a_chunk2 = results_a[1][2].chunk_hash
    hash_b_chunk2 = results_b[1][2].chunk_hash

    print(f"Prefix A chunk 2 hash: {hash_a_chunk2}")
    print(f"Prefix B chunk 2 hash: {hash_b_chunk2}")

    if hash_a_chunk2 == hash_b_chunk2:
        print("\n✓ PASS: Same content -> Same hash!")
        return True
    else:
        print("\n✗ FAIL: Different hashes (still using prefix chain)")
        return False

test_content_only_hashing()

## 6. Test vLLM with LMCache

In [ ]:
from vllm import LLM, SamplingParams

# Load a small model
llm = LLM(
    model="Qwen/Qwen2-0.5B",
    max_model_len=512,
    enforce_eager=True,
)

print("Model loaded!")

In [ ]:
# Test basic generation
prompts = [
    "Hello, my name is",
    "The capital of France is",
]

sampling_params = SamplingParams(temperature=0.7, max_tokens=50)
outputs = llm.generate(prompts, sampling_params)

for output in outputs:
    print(f"Prompt: {output.prompt}")
    print(f"Generated: {output.outputs[0].text}")
    print()

## 7. Test Selective Loading with LMCache

This requires enabling LMCache in vLLM. Start the server separately.

In [ ]:
# Write LMCache config
%%writefile lmcache_config.yaml
chunk_size: 256
local_cpu: true
max_local_cpu_size: 1073741824  # 1GB

In [ ]:
# Start vLLM server with LMCache in background
import subprocess
import time

# Start server
server = subprocess.Popen([
    "python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", "Qwen/Qwen2-0.5B",
    "--max-model-len", "512",
    "--enable-prefix-caching",
    "--port", "8000",
], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

print("Starting vLLM server...")
time.sleep(30)  # Wait for server to start
print("Server should be ready!")

In [ ]:
# Test with OpenAI client
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="dummy")

# First request - cache the conversation
response = client.chat.completions.create(
    model="Qwen/Qwen2-0.5B",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "My name is Alice and I love coffee."},
        {"role": "assistant", "content": "Nice to meet you Alice! I'll remember you love coffee."},
        {"role": "user", "content": "What's my name?"},
    ],
    max_tokens=50,
)

print("Response:", response.choices[0].message.content)

In [ ]:
# Test selective loading (if adapter supports it)
# This uses the extra_body parameter to pass selective hashes

import hashlib

def make_uuid(text):
    return int(hashlib.sha256(text.encode()).hexdigest()[:16], 16)

# UUIDs for specific blocks we want to load
system_uuid = make_uuid("system:You are a helpful assistant.")
coffee_uuid = make_uuid("user:My name is Alice and I love coffee.")

try:
    response = client.chat.completions.create(
        model="Qwen/Qwen2-0.5B",
        messages=[
            {"role": "user", "content": "What do I like to drink?"},
        ],
        max_tokens=50,
        extra_body={
            "kv_transfer_params": {
                "lmcache.selective_hashes": [system_uuid, coffee_uuid],
                "lmcache.selective_offsets": [256, 256],
            }
        }
    )
    print("Selective load response:", response.choices[0].message.content)
except Exception as e:
    print(f"Selective loading not yet integrated: {e}")

In [ ]:
# Cleanup - stop the server
server.terminate()
print("Server stopped")

## Summary

The selective KV cache loading works by:

1. **Content-only hashing** - Each block's hash depends only on its content, not previous blocks
2. **UUID-based retrieval** - Load blocks by hash instead of requiring all prefix tokens
3. **Contiguous mapping** - Selected blocks are placed at positions 0, 1, 2... regardless of original positions

This enables:
- Loading only relevant conversation turns
- Skipping irrelevant context
- Faster inference with selective memory